In [46]:
using LowLevelFEM, LinearAlgebra

In [47]:
p = 5

5

In [48]:
structured_rect_mesh(n=10, order=p)

mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, field=:u);

In [49]:
k = mat.k

s = ScalarField(U, "right", (x, y, z)->y > 0.5 ? 1 : 10)

K = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U))
f = ∫(U ⋅ [s, 0], Γ="right")

bc = BoundaryCondition("left", ux=0, uy=0);

In [50]:
fixed = constrainedDoFs(U, [bc])
free = freeDoFs(U, [bc]);

In [51]:
u1 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u1.a[fixed, 1]
u1.a[free] = (K.A[free, free]) \ (f.a[free, 1] - f_kin[free, 1]);

In [52]:
showDoFResults(u1, name="u1", visible=true);

In [53]:
T, R = reductionMatrices(U);

In [54]:
u2 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u2.a[fixed, 1]
Kr = T[free, :]' * K.A[free, free] * T[free, :]
fr = T[free, :]' * (f.a[free, 1] - f_kin[free, 1])

ur = Kr \ fr

u2.a[free] = (T*ur)[free];

In [55]:
showDoFResults(u2, name="u2", visible=true);

In [56]:
norm(u2.a - u1.a) / norm(u1.a)

0.0001083922188951281

In [57]:
∫(U, "body", u1[1])

1.7219083782371756

In [58]:
∫(U, "body", u2[1])

1.7218139232524423

In [59]:
openPostProcessor();